# Load an existing RayJob and check its status

This notebook attaches to a RayJob that already exists in a Kubernetes
cluster and reads its normalized status. It runs entirely offline: a small
in-kernel fake cluster client stands in for the Kubernetes API, so the
example needs no real cluster and no network access.


In [ ]:
%load_ext tau.widgets.ipython

In [ ]:
from tau.widgets.panel import TauGridPanel
from tau.widgets.kube import ClusterClient
from tau.widgets.status import list_runs

RAYJOB = {
    "metadata": {
        "name": "demo-rayjob",
        "labels": {"kueue.x-k8s.io/queue-name": "research-gpu"},
    },
    "status": {
        "jobStatus": "RUNNING",
        "rayClusterName": "demo-rayjob-raycluster",
        "jobId": "raysubmit_123",
        "jobDeploymentStatus": "Running",
        "conditions": [{"type": "Admitted", "status": "True"}],
    },
}

PODS = {
    "items": [
        {
            "metadata": {
                "name": "demo-rayjob-head-abc",
                "labels": {"ray.io/node-type": "head"},
            },
            "spec": {"nodeName": "gpu-node-1"},
            "status": {
                "phase": "Running",
                "conditions": [{"type": "Ready", "status": "True"}],
                "containerStatuses": [{"restartCount": 1, "ready": True}],
            },
        }
    ]
}


class FakeCustomApi:
    """Offline stand-in for the Kubernetes CustomObjectsApi."""

    def __init__(self, rayjob=RAYJOB):
        self.rayjob = rayjob

    def get_namespaced_custom_object(self, **kwargs):
        return self.rayjob

    def list_namespaced_custom_object(self, **kwargs):
        return {"items": [self.rayjob] if self.rayjob else []}


class FakeCoreApi:
    """Offline stand-in for the Kubernetes CoreV1Api pod listing."""

    def __init__(self, pods=PODS["items"]):
        self.pods = pods

    def list_namespaced_pod(self, namespace, label_selector=None, **kwargs):
        return {"items": self.pods}


def build_panel():
    """Return a TauGridPanel wired to the in-kernel fake cluster."""
    client = ClusterClient(custom=FakeCustomApi(), core=FakeCoreApi())
    return TauGridPanel(namespace="ray", client=client)


In [ ]:
panel = build_panel()
status = panel.load('demo-rayjob', namespace='ray')
print(status.state, status.ready_pods, status.total_pods, status.queue)
assert status.existing is True and status.state == 'running' and status.ready_pods == 1 and status.total_pods == 1

In [ ]:
status = panel.check_status()
print([d.code for d in status.diagnostics])

In [ ]:
panel

In [ ]:
runs = list_runs(panel.client, namespace='ray')
print([run.name for run in runs])